# Compression Strain Analysis

Run each cell **in order from top to bottom**.

> **First time only (Colab):** After Cell 1 finishes installing packages, go to **Runtime → Restart runtime**, then run all cells again from Cell 2 onward.

In [ ]:
# ── CELL 1: Install packages ─────────────────────────────
# Only needs to run once per Colab session.
import subprocess
subprocess.run(['pip', 'install', 'tifffile', '-q'], check=True)
print('All packages ready! Continue to Cell 2.')

In [ ]:
# ── CELL 2: Imports ──────────────────────────────────────
import numpy as np
import tifffile as tiff
import os
import cv2
import csv
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.path import Path
from scipy.spatial import Delaunay
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from google.colab import files as colab_files
    IN_COLAB = True
    print('Running in Google Colab')
except ImportError:
    IN_COLAB = False
    print('Running locally (Windows/Mac)')

print('Imports complete — continue to Cell 3.')

## Step 1 — Settings

Adjust the values below, then run **Cell 4** to upload your file.

In [ ]:
# ── CELL 3: Settings ─────────────────────────────────────
s  = {'description_width': '170px'}
lw = widgets.Layout(width='360px')

start_frame_w    = widgets.IntText(value=8,     description='Start Frame:',            style=s, layout=lw)
end_frame_w      = widgets.Text(value='',        description='End Frame (blank = all):', style=s, layout=lw)
frame_skip_w     = widgets.IntText(value=2,      description='Frame Skip:',             style=s, layout=lw)
strain_mode_w    = widgets.Dropdown(options=['vonmises','ex','ey','gxy'], value='vonmises',
                                    description='Strain Mode:', style=s, layout=lw)
mesh_spacing_w   = widgets.IntText(value=25,     description='Mesh Spacing (px):',      style=s, layout=lw)
show_edges_w     = widgets.Checkbox(value=True,  description='Show Triangle Edges')
delta_x_w        = widgets.IntText(value=40,     description='Template Width:',         style=s, layout=lw)
delta_y_w        = widgets.IntText(value=40,     description='Template Height:',        style=s, layout=lw)
t_x_w            = widgets.IntText(value=15,     description='Search Range X:',         style=s, layout=lw)
t_y_w            = widgets.IntText(value=15,     description='Search Range Y:',         style=s, layout=lw)
corr_threshold_w = widgets.FloatText(value=0.85, description='Corr Threshold:',         style=s, layout=lw)
max_disp_w       = widgets.FloatText(value=12.0, description='Max Displacement:',       style=s, layout=lw)
gaussian_blur_w  = widgets.IntText(value=3,      description='Gaussian Blur (0 = off):', style=s, layout=lw)
low_in_w         = widgets.IntText(value=28,     description='Low In:',                 style=s, layout=lw)
high_in_w        = widgets.IntText(value=250,    description='High In:',                style=s, layout=lw)

left_col  = widgets.VBox([
    widgets.HTML('<b>— Frames —</b>'),  start_frame_w, end_frame_w, frame_skip_w,
    widgets.HTML('<b>— Strain —</b>'),  strain_mode_w,
    widgets.HTML('<b>— Mesh —</b>'),    mesh_spacing_w, show_edges_w,
    widgets.HTML('<b>— Contrast —</b>'), low_in_w, high_in_w,
])
right_col = widgets.VBox([
    widgets.HTML('<b>— Tracking —</b>'),
    delta_x_w, delta_y_w, t_x_w, t_y_w,
    corr_threshold_w, max_disp_w, gaussian_blur_w,
])
display(widgets.HBox([left_col, widgets.HTML('&nbsp;&nbsp;&nbsp;&nbsp;'), right_col]))
print('Adjust settings above, then run Cell 4 to upload your file.')

## Step 2 — Upload File

Run Cell 4. On **Colab** a file picker will appear. On **Windows/Mac** a system file dialog will open.

In [ ]:
# ── CELL 4: File selection ───────────────────────────────
if IN_COLAB:
    print('Click "Choose Files" below to upload your image stack or video:')
    uploaded = colab_files.upload()
    if not uploaded:
        raise RuntimeError('No file uploaded.')
    filename = list(uploaded.keys())[0]
else:
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()
    root.lift()
    root.attributes('-topmost', True)
    filename = filedialog.askopenfilename(
        title='Select image stack or video',
        filetypes=[
            ('All supported', '*.tiff *.tif *.mov *.mp4 *.avi *.mkv'),
            ('TIFF stacks',   '*.tiff *.tif'),
            ('Video files',   '*.mov *.mp4 *.avi *.mkv'),
            ('All files',     '*.*'),
        ]
    )
    root.destroy()
    if not filename:
        raise RuntimeError('No file selected.')

print(f'File selected: {filename}')
print('Run Cell 5 to load and preprocess the images.')

## Step 3 — Load & Preprocess Images

In [ ]:
# ── CELL 5: Load & preprocess ────────────────────────────
start_frame      = start_frame_w.value
end_frame        = None if end_frame_w.value.strip() == '' else int(end_frame_w.value.strip())
strain_mode      = strain_mode_w.value
mesh_spacing     = mesh_spacing_w.value
show_tri_edges   = show_edges_w.value
delta_x          = delta_x_w.value
delta_y          = delta_y_w.value
t_x              = t_x_w.value
t_y              = t_y_w.value
low_in           = low_in_w.value
high_in          = high_in_w.value
frame_skip       = frame_skip_w.value
corr_threshold   = corr_threshold_w.value
max_disp         = max_disp_w.value
gaussian_blur    = gaussian_blur_w.value

ext = os.path.splitext(filename)[1].lower()
if ext in ('.tiff', '.tif'):
    imgs = tiff.imread(filename)
    if imgs.ndim == 4:
        imgs = np.array([cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in imgs])
    if imgs.ndim != 3:
        raise ValueError('TIFF must be a grayscale image stack.')
else:
    cap = cv2.VideoCapture(filename)
    if not cap.isOpened():
        raise ValueError(f'Cannot open video file: {filename}')
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    cap.release()
    if not frames:
        raise ValueError(f'No frames read from: {filename}')
    imgs = np.array(frames)

if frame_skip > 1:
    imgs = imgs[::frame_skip]
if end_frame is not None:
    imgs = imgs[:end_frame + 1]

Nf, mm, nn = imgs.shape

def adjust(img):
    img = np.clip((img - low_in) / (high_in - low_in), 0, 1)
    return (img * 255).astype(np.uint8)

imgs = np.array([adjust(f) for f in imgs])

if gaussian_blur > 0:
    ksize = gaussian_blur if gaussian_blur % 2 == 1 else gaussian_blur + 1
    imgs = np.array([cv2.GaussianBlur(f, (ksize, ksize), 0) for f in imgs])

print(f'Loaded {Nf} frames, image size {mm} x {nn} pixels')
print(f'Reference (start) frame: {start_frame}')
print('Run Cell 6 to draw the region of interest.')

## Step 4 — Draw Region of Interest

1. Run **Cell 6** — your image will appear with an interactive canvas
2. Click around your specimen to place polygon points (green dots will appear)
3. Use **Undo Last** or **Clear All** to fix mistakes
4. Click **Finish ROI** when done
5. Run **Cell 6b** to confirm the points were captured

In [ ]:
# ── CELL 6: Draw ROI ─────────────────────────────────────
import base64, io, json
from IPython.display import display, HTML

if IN_COLAB:
    from google.colab import output as colab_output
    from PIL import Image as PILImage

    # Register a Python callback — JS will call this when Finish ROI is clicked
    _roi_points = []
    def _receive_roi(pts_json):
        global _roi_points
        _roi_points = json.loads(pts_json)
    colab_output.register_callback('notebook._receiveROI', _receive_roi)

    pil_img = PILImage.fromarray(imgs[start_frame])
    buf = io.BytesIO()
    pil_img.save(buf, format='PNG')
    img_b64 = base64.b64encode(buf.getvalue()).decode()

    html = f"""
    <p style="font-family:monospace;font-size:13px;margin:0 0 6px 0">
        <b>Click on the image to place points around your specimen, then click Finish ROI.</b>
    </p>
    <canvas id="roiCanvas" style="max-width:700px;width:100%;cursor:crosshair;
                                   border:1px solid #666;display:block;"></canvas>
    <div id="roiInfo" style="font-family:monospace;font-size:13px;margin:6px 0">
        0 point(s) — click to start
    </div>
    <button onclick="roiFinish()"
            style="padding:6px 14px;margin:0 6px 0 0;background:#4CAF50;
                   color:white;border:none;border-radius:4px;cursor:pointer;font-size:13px;">
        Finish ROI
    </button>
    <button onclick="roiUndo()"
            style="padding:6px 14px;margin:0 6px 0 0;background:#FF9800;
                   color:white;border:none;border-radius:4px;cursor:pointer;font-size:13px;">
        Undo Last
    </button>
    <button onclick="roiClear()"
            style="padding:6px 14px;margin:0;background:#f44336;
                   color:white;border:none;border-radius:4px;cursor:pointer;font-size:13px;">
        Clear All
    </button>
    <script>
    (function() {{
        var pts  = [];
        var done = false;

        const canvas = document.getElementById('roiCanvas');
        const ctx    = canvas.getContext('2d');
        const info   = document.getElementById('roiInfo');
        const img    = new Image();

        img.onload = function() {{
            canvas.width  = img.naturalWidth;
            canvas.height = img.naturalHeight;
            ctx.drawImage(img, 0, 0);
        }};
        img.src = 'data:image/png;base64,{img_b64}';

        function redraw() {{
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            ctx.drawImage(img, 0, 0);
            if (!pts.length) return;
            ctx.strokeStyle = '#39FF14'; ctx.fillStyle = '#39FF14'; ctx.lineWidth = 2;
            ctx.beginPath(); ctx.moveTo(pts[0][0], pts[0][1]);
            pts.forEach(function(p) {{ ctx.lineTo(p[0], p[1]); }});
            ctx.closePath(); ctx.stroke();
            pts.forEach(function(p) {{
                ctx.beginPath(); ctx.arc(p[0], p[1], 5, 0, 2*Math.PI); ctx.fill();
            }});
        }}

        canvas.addEventListener('click', function(e) {{
            if (done) return;
            const r  = canvas.getBoundingClientRect();
            pts.push([(e.clientX - r.left) * canvas.width  / r.width,
                       (e.clientY - r.top)  * canvas.height / r.height]);
            info.textContent = pts.length + ' point(s) placed';
            redraw();
        }});

        window.roiUndo = function() {{
            if (!done) {{ pts.pop(); info.textContent = pts.length + ' point(s)'; redraw(); }}
        }};
        window.roiClear = function() {{
            if (!done) {{ pts = []; info.textContent = '0 point(s)'; redraw(); }}
        }};
        window.roiFinish = function() {{
            if (pts.length < 3) {{ info.textContent = '⚠ Need at least 3 points!'; return; }}
            done = true;
            canvas.style.cursor = 'default';
            info.textContent = '✓ ROI sent to Python — now run Cell 6b.';
            google.colab.kernel.invokeFunction('notebook._receiveROI', [JSON.stringify(pts)], {{}});
        }};
    }})();
    </script>
    """
    display(HTML(html))
    print("Draw your ROI above, click 'Finish ROI', then run Cell 6b.")

else:
    from IPython import get_ipython
    get_ipython().run_line_magic('matplotlib', 'notebook')

    polygon_vertices = []
    roi_done = [False]

    fig_roi, ax_roi = plt.subplots(figsize=(8, 6))
    ax_roi.imshow(imgs[start_frame], cmap='gray')
    ax_roi.set_title('Click to place points | Use buttons when done')
    line_plot,   = ax_roi.plot([], [], color='#39FF14', linewidth=2)
    points_plot, = ax_roi.plot([], [], 'o', color='#39FF14', markersize=5)

    finish_btn = widgets.Button(description='Finish ROI',      button_style='success')
    undo_btn   = widgets.Button(description='Undo Last Point', button_style='warning')
    clear_btn  = widgets.Button(description='Clear All',       button_style='danger')
    status_lbl = widgets.Label(value='0 points placed')

    def _update_plot():
        if polygon_vertices:
            xs, ys = zip(*polygon_vertices)
            line_plot.set_data(xs + (xs[0],), ys + (ys[0],))
            points_plot.set_data(xs, ys)
        else:
            line_plot.set_data([], [])
            points_plot.set_data([], [])
        fig_roi.canvas.draw_idle()

    def _onclick(event):
        if event.inaxes != ax_roi or roi_done[0]: return
        polygon_vertices.append((event.xdata, event.ydata))
        status_lbl.value = f'{len(polygon_vertices)} point(s) placed'
        _update_plot()

    def _on_undo(b):
        if polygon_vertices and not roi_done[0]:
            polygon_vertices.pop()
            status_lbl.value = f'{len(polygon_vertices)} point(s) placed'
            _update_plot()

    def _on_clear(b):
        if not roi_done[0]:
            polygon_vertices.clear()
            status_lbl.value = '0 points placed'
            _update_plot()

    def _on_finish(b):
        if len(polygon_vertices) < 3:
            status_lbl.value = 'Need at least 3 points!'
            return
        roi_done[0] = True
        status_lbl.value = f'ROI set with {len(polygon_vertices)} points. Run Cell 6b.'
        finish_btn.disabled = undo_btn.disabled = clear_btn.disabled = True

    fig_roi.canvas.mpl_connect('button_press_event', _onclick)
    finish_btn.on_click(_on_finish)
    undo_btn.on_click(_on_undo)
    clear_btn.on_click(_on_clear)
    display(widgets.HBox([finish_btn, undo_btn, clear_btn, status_lbl]))
    plt.tight_layout()
    plt.show()

In [ ]:
# ── CELL 6b: Confirm ROI ─────────────────────────────────
# Run this AFTER clicking "Finish ROI" in Cell 6.
if IN_COLAB:
    polygon_vertices = [(p[0], p[1]) for p in _roi_points]
    roi_done = [True]
    if len(polygon_vertices) < 3:
        raise RuntimeError(
            f'Only {len(polygon_vertices)} point(s) received. '
            'Go back to Cell 6, place at least 3 points, click Finish ROI, then re-run this cell.'
        )
    print(f'ROI confirmed: {len(polygon_vertices)} points. Run Cell 7.')
else:
    if not roi_done[0] or len(polygon_vertices) < 3:
        raise RuntimeError('ROI not finished. Go back to Cell 6 and click Finish ROI.')
    print(f'ROI confirmed: {len(polygon_vertices)} points. Run Cell 7.')

## Step 5 — Run Analysis

This may take a few minutes depending on the number of frames and mesh density.

In [ ]:
# ── CELL 7: Mesh + Tracking + Strain ─────────────────────
if not roi_done[0] or len(polygon_vertices) < 3:
    raise RuntimeError('ROI not finished. Go back to Cell 6 and click Finish ROI.')

# Build mesh grid
roi_path = Path(polygon_vertices)
poly_arr = np.array(polygon_vertices)
xmin, ymin = np.min(poly_arr, axis=0)
xmax, ymax = np.max(poly_arr, axis=0)

pts = []
for r in np.arange(int(ymin), int(ymax), mesh_spacing):
    for c in np.arange(int(xmin), int(xmax), mesh_spacing):
        if roi_path.contains_point((c, r)):
            pts.append((int(r), int(c)))

Nx = len(pts)
print(f'Mesh points: {Nx}')

XA = np.full((Nx, Nf), np.nan)
YA = np.full((Nx, Nf), np.nan)
for i, (r, c) in enumerate(pts):
    XA[i, start_frame] = r
    YA[i, start_frame] = c

tri = Delaunay(np.array([[p[1], p[0]] for p in pts]))

# Tracking function
def track(img1, img2, x, y):
    target = img1[x:x+delta_x, y:y+delta_y]
    scan   = img2[x-t_x:x+delta_x+t_x, y-t_y:y+delta_y+t_y]
    if target.size == 0 or scan.shape[0] < delta_x or scan.shape[1] < delta_y:
        return 0, 0
    if np.all(target == 0):
        return 0, 0
    result = cv2.matchTemplate(scan, target, cv2.TM_CCOEFF_NORMED)
    _, max_val, _, max_loc = cv2.minMaxLoc(result)
    if max_val < corr_threshold:
        return 0, 0
    mx, my = max_loc[1], max_loc[0]
    h, w = result.shape
    sub_mx, sub_my = float(mx), float(my)
    if 0 < mx < h - 1:
        a, b, c = result[mx-1, my], result[mx, my], result[mx+1, my]
        denom = 2*b - a - c
        if abs(denom) > 1e-12:
            sub_mx = mx + (a - c) / (2 * denom)
    if 0 < my < w - 1:
        a, b, c = result[mx, my-1], result[mx, my], result[mx, my+1]
        denom = 2*b - a - c
        if abs(denom) > 1e-12:
            sub_my = my + (a - c) / (2 * denom)
    dx = -t_x + sub_mx
    dy = -t_y + sub_my
    if dx*dx + dy*dy > max_disp*max_disp:
        return 0, 0
    return dx, dy

# Track frame by frame
total_frames = Nf - 1 - start_frame
for k in range(start_frame, Nf - 1):
    if (k - start_frame) % max(1, total_frames // 10) == 0:
        print(f'  Tracking frame {k - start_frame + 1} / {total_frames} ...')
    for i in range(Nx):
        x = int(XA[i, k])
        y = int(YA[i, k])
        x = max(t_x, min(x, mm - delta_x - t_x - 1))
        y = max(t_y, min(y, nn - delta_y - t_y - 1))
        dx, dy = track(imgs[k], imgs[k+1], x, y)
        XA[i, k+1] = x + dx
        YA[i, k+1] = y + dy

# Compute strain (CST finite element formulation)
strain = np.full((Nf, len(tri.simplices), 3), np.nan)
for k in range(start_frame, Nf):
    for t, s in enumerate(tri.simplices):
        n1, n2, n3 = s
        x1r, y1r = YA[n1, start_frame], XA[n1, start_frame]
        x2r, y2r = YA[n2, start_frame], XA[n2, start_frame]
        x3r, y3r = YA[n3, start_frame], XA[n3, start_frame]
        x1,  y1  = YA[n1, k], XA[n1, k]
        x2,  y2  = YA[n2, k], XA[n2, k]
        x3,  y3  = YA[n3, k], XA[n3, k]
        if np.isnan(x1) or np.isnan(x2) or np.isnan(x3):
            continue
        u = np.array([x1-x1r, y1-y1r, x2-x2r, y2-y2r, x3-x3r, y3-y3r])
        x13, x23 = x1r-x3r, x2r-x3r
        y13, y23 = y1r-y3r, y2r-y3r
        det = x13*y23 - y13*x23
        if abs(det) < 1e-8:
            continue
        B = (1/det) * np.array([
            [y23, 0, y3r-y1r, 0, y1r-y2r, 0],
            [0, x3r-x2r, 0, x1r-x3r, 0, x2r-x1r],
            [x3r-x2r, y23, x1r-x3r, y3r-y1r, x2r-x1r, y1r-y2r]
        ])
        strain[k, t, :] = B @ u

print('Analysis complete! Run Cell 8 to export results.')

## Step 6 — Export Results

Saves three output files to a timestamped folder inside `results/`:
- **CSV** — per-triangle strain values (`ex`, `ey`, `gxy`) for every frame
- **Plot** — mean strain over time as a PNG image
- **Video** — strain heatmap overlaid on the original frames as an MP4

On **Google Colab**, all three files are automatically zipped into a single `.zip` and downloaded to your computer. Allow the download if your browser asks. If nothing downloads, open the folder icon in the left sidebar, navigate to `results/`, and right-click any file to download manually.

On **local Jupyter / VS Code**, files are saved directly to the `results/` folder — no download step needed.

In [ ]:
# ── CELL 8: Export ───────────────────────────────────────
%matplotlib inline

base_name  = os.path.splitext(os.path.basename(filename))[0]
timestamp  = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
export_dir = os.path.join(os.getcwd(), 'results', f'{base_name}_{timestamp}')
os.makedirs(export_dir, exist_ok=True)

def get_plot_data(mode):
    if mode == 'ey':
        return strain[:,:,1], 'εy (Vertical Strain)'
    elif mode == 'ex':
        return strain[:,:,0], 'εx (Horizontal Strain)'
    elif mode == 'gxy':
        return strain[:,:,2], 'γxy (Shear Strain)'
    else:
        ex, ey, g = strain[:,:,0], strain[:,:,1], strain[:,:,2]
        return np.sqrt(ex**2 - ex*ey + ey**2 + 0.75*g**2), 'Von Mises Strain'

export_data, plot_label = get_plot_data(strain_mode)

# CSV
csv_path = os.path.join(export_dir, f'{base_name}_strain.csv')
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['frame', 'triangle', 'ex', 'ey', 'gxy'])
    for k in range(start_frame, Nf):
        for t in range(len(tri.simplices)):
            ex_v, ey_v, gxy_v = strain[k, t, :]
            if not np.isnan(ex_v):
                writer.writerow([k, t, f'{ex_v:.6f}', f'{ey_v:.6f}', f'{gxy_v:.6f}'])
print('CSV saved.')

# Mean strain over time plot
mean_strain = np.nanmean(export_data, axis=1)
fig_plot, ax_plot = plt.subplots(figsize=(8, 4))
ax_plot.plot(np.arange(Nf)[start_frame:], mean_strain[start_frame:], linewidth=1.5)
ax_plot.set_xlabel('Frame')
ax_plot.set_ylabel(f'Mean {plot_label}')
ax_plot.set_title('Strain Over Time')
ax_plot.grid(True, alpha=0.3)
fig_plot.tight_layout()
plot_path = os.path.join(export_dir, f'{base_name}_strain_over_time.png')
fig_plot.savefig(plot_path, dpi=150)
plt.show()
print('Plot saved.')

# Export video
vmin = np.nanpercentile(export_data, 5)
vmax = np.nanpercentile(export_data, 95)
cmap = plt.cm.jet
video_path = os.path.join(export_dir, f'{base_name}_strain.mp4')
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_out = cv2.VideoWriter(video_path, fourcc, 15, (nn, mm))
for k in range(start_frame, Nf):
    if (k - start_frame) % max(1, (Nf - start_frame) // 10) == 0:
        print(f'  Exporting frame {k - start_frame + 1} / {Nf - start_frame} ...')
    img = cv2.cvtColor(imgs[k], cv2.COLOR_GRAY2RGB)
    overlay = img.copy()
    pts_now = np.array([[YA[i, k], XA[i, k]] for i in range(Nx)])
    for t, s in enumerate(tri.simplices):
        val = export_data[k, t]
        if np.isnan(val):
            continue
        n = (val - vmin) / (vmax - vmin + 1e-12)
        color = cmap(n)[:3]
        color = tuple(int(255 * c) for c in color)
        poly_pts = pts_now[s].astype(int)
        cv2.fillPoly(overlay, [poly_pts], color)
        if show_tri_edges:
            cv2.polylines(overlay, [poly_pts], True, (0, 0, 0), 1)
    frame_out = cv2.addWeighted(overlay, 0.45, img, 0.55, 0)
    frame_out = cv2.cvtColor(frame_out, cv2.COLOR_RGB2BGR)
    video_out.write(frame_out)
video_out.release()
print('Video saved.')

# Zip all results into one file and download (Colab only)
if IN_COLAB:
    import zipfile
    zip_path = os.path.join(os.getcwd(), 'results', f'{base_name}_{timestamp}.zip')
    with zipfile.ZipFile(zip_path, 'w') as zf:
        zf.write(csv_path,   os.path.basename(csv_path))
        zf.write(plot_path,  os.path.basename(plot_path))
        zf.write(video_path, os.path.basename(video_path))
    print('\nDownloading results as a single zip file...')
    colab_files.download(zip_path)
    print('Done! Run Cell 9 to open the interactive viewer.')
else:
    print(f'\nAll files saved to: {export_dir}')
    print('Run Cell 9 to open the interactive viewer.')

## Step 7 — Interactive Viewer

Browse the strain field frame by frame using the slider below. The color overlay uses a jet colormap (blue = low strain, red = high strain), scaled to the 5th–95th percentile of strain values across all frames.

Drag the **Frame** slider left or right to step through the analysis. The colorbar on the right shows the strain scale for the selected mode.

In [ ]:
# ── CELL 9: Interactive viewer ───────────────────────────
%matplotlib inline

data, label = get_plot_data(strain_mode)
vmin_v = np.nanpercentile(data, 5)
vmax_v = np.nanpercentile(data, 95)
cmap_v = plt.cm.jet
norm_v = mpl.colors.Normalize(vmin=vmin_v, vmax=vmax_v)

slider = widgets.IntSlider(
    value=start_frame, min=start_frame, max=Nf - 1, step=1,
    description='Frame:',
    layout=widgets.Layout(width='550px'),
    style={'description_width': '60px'}
)
out_view = widgets.Output()

def show_frame(k):
    with out_view:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 6))
        img = cv2.cvtColor(imgs[k], cv2.COLOR_GRAY2RGB)
        overlay = img.copy()
        pts_now = np.array([[YA[i, k], XA[i, k]] for i in range(Nx)])
        for t, s in enumerate(tri.simplices):
            val = data[k, t]
            if np.isnan(val):
                continue
            n = (val - vmin_v) / (vmax_v - vmin_v + 1e-12)
            color = cmap_v(n)[:3]
            color = tuple(int(255 * c) for c in color)
            poly_pts = pts_now[s].astype(int)
            cv2.fillPoly(overlay, [poly_pts], color)
            if show_tri_edges:
                cv2.polylines(overlay, [poly_pts], True, (0, 0, 0), 1)
        img_out = cv2.addWeighted(overlay, 0.45, img, 0.55, 0)
        ax.imshow(img_out)
        sm = mpl.cm.ScalarMappable(cmap=cmap_v, norm=norm_v)
        sm.set_array([])
        fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04, label=label)
        ax.set_title(f'Frame {k}  |  {label}')
        ax.axis('off')
        plt.tight_layout()
        plt.show()

slider.observe(lambda change: show_frame(change['new']), names='value')
display(slider, out_view)
show_frame(start_frame)